# Amazon iPhone 17 Campaign — RFM Segmentation & Geo Analytics

**Story ID:** STORY-003  
**Business Goal:** Identify high-value customers for a targeted iPhone 17 campaign using RFM segmentation and geo analytics. Output feeds Amazon's marketing automation platform.

This notebook walks through the PySpark pipeline that the Autonomous ETL Agent generates when given STORY-003. You can run it interactively on a Dataproc cluster or locally with a small dataset.

## RFM Explained

| Metric | Definition | Why it matters |
|---|---|---|
| **Recency** | Days since last purchase | Recent buyers are more likely to convert |
| **Frequency** | Number of orders in window | Loyal customers have higher LTV |
| **Monetary** | Total spend in window | High spenders → premium product targets |

Customers are scored 1–4 per metric using **quartile-based bucketing** (`ntile(4)`).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, DateType
)
import datetime

# Local mode for the notebook. On Dataproc the SparkSession is pre-configured.
spark = (
    SparkSession.builder
    .appName("amazon_campaign_rfm_story_003")
    .config("spark.sql.shuffle.partitions", "8")   # small for local
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Sample Data

In production these come from GCS (`gs://YOUR_PROJECT-raw/transactions/` and `gs://YOUR_PROJECT-raw/customer_profiles/`).  
For the notebook we seed synthetic data inline.

In [ ]:
# ── Synthetic Dataset ─────────────────────────────────────────────────────────
today = datetime.date(2025, 6, 1)

transactions_data = [
    ("C001", "ORD-101", "2025-05-28",  320.00, "tech"),
    ("C001", "ORD-102", "2025-05-15",  150.00, "books"),
    ("C001", "ORD-103", "2025-04-10",  420.00, "tech"),
    ("C002", "ORD-201", "2025-05-30", 1200.00, "tech"),
    ("C002", "ORD-202", "2025-05-20",  850.00, "tech"),
    ("C003", "ORD-301", "2025-01-05",   45.00, "home"),
    ("C004", "ORD-401", "2025-05-25",  680.00, "tech"),
    ("C004", "ORD-402", "2025-05-10",  200.00, "clothing"),
    ("C004", "ORD-403", "2025-03-15",  310.00, "tech"),
    ("C004", "ORD-404", "2025-02-20",  190.00, "books"),
    ("C005", "ORD-501", "2025-04-01",  560.00, "tech"),
    ("C005", "ORD-502", "2025-03-20",  120.00, "home"),
    ("C006", "ORD-601", "2024-11-11",   30.00, "clothing"),
    ("C007", "ORD-701", "2025-05-29",  999.00, "tech"),
    ("C007", "ORD-702", "2025-05-22",  450.00, "tech"),
    ("C007", "ORD-703", "2025-05-01",  275.00, "books"),
    ("C007", "ORD-704", "2025-04-15",  620.00, "tech"),
]

txn_schema = StructType([
    StructField("customer_id",    StringType(), False),
    StructField("order_id",       StringType(), False),
    StructField("order_date",     StringType(), False),
    StructField("order_value",    DoubleType(), False),
    StructField("product_category", StringType(), False),
])

profiles_data = [
    ("C001", "Alice",   "West",     "active"),
    ("C002", "Bob",     "East",     "active"),
    ("C003", "Charlie", "South",    "inactive"),
    ("C004", "Diana",   "West",     "active"),
    ("C005", "Eve",     "Midwest",  "active"),
    ("C006", "Frank",   "East",     "inactive"),
    ("C007", "Grace",   "West",     "active"),
]

prof_schema = StructType([
    StructField("customer_id",    StringType(), False),
    StructField("customer_name",  StringType(), True),
    StructField("shipping_region", StringType(), True),
    StructField("account_status", StringType(), True),
])

df_txn  = spark.createDataFrame(transactions_data, txn_schema)
df_prof = spark.createDataFrame(profiles_data, prof_schema)

df_txn  = df_txn.withColumn("order_date", F.to_date("order_date"))
print("Transactions:", df_txn.count())
df_txn.show(5, truncate=False)

## 2. RFM Score Calculation

We compute raw R/F/M values per customer over the trailing 90 days (configurable via `LOOKBACK_DAYS`), then bucket each into quartiles 1–4 using `ntile(4)` window functions.

- **Recency score**: higher quartile = more recent *(invert the days value)*
- **Frequency score**: higher quartile = more orders
- **Monetary score**: higher quartile = higher total spend

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LOOKBACK_DAYS = 90
snapshot_date = F.lit(today).cast(DateType())
cutoff_date   = F.date_sub(snapshot_date, LOOKBACK_DAYS)

# Filter to active customers in the lookback window
df_active_txn = (
    df_txn
    .filter(F.col("order_date") >= cutoff_date)
    .join(df_prof.filter(F.col("account_status") == "active"), "customer_id", "inner")
)

# ── Raw RFM Aggregation ───────────────────────────────────────────────────────
df_rfm_raw = (
    df_active_txn
    .groupBy("customer_id", "shipping_region")
    .agg(
        F.datediff(snapshot_date, F.max("order_date")).alias("days_since_last_order"),
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.sum("order_value"), 2).alias("total_spend"),
    )
)

print("Raw RFM values:")
df_rfm_raw.show(truncate=False)

In [ ]:
# ── Quartile Scoring via ntile Window ────────────────────────────────────────
w_global = Window.orderBy(F.lit(1))  # global ranking

df_rfm_scored = (
    df_rfm_raw
    .withColumn(
        # Recency: lower days_since_last_order = better → sort ascending, invert quartile
        "recency_score",
        5 - F.ntile(4).over(Window.orderBy(F.col("days_since_last_order").asc()))
    )
    .withColumn(
        "frequency_score",
        F.ntile(4).over(Window.orderBy(F.col("order_count").asc()))
    )
    .withColumn(
        "monetary_score",
        F.ntile(4).over(Window.orderBy(F.col("total_spend").asc()))
    )
    .withColumn(
        "rfm_total",
        F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score")
    )
)

print("RFM Scores:")
df_rfm_scored.select(
    "customer_id", "days_since_last_order", "order_count", "total_spend",
    "recency_score", "frequency_score", "monetary_score", "rfm_total"
).orderBy(F.col("rfm_total").desc()).show(truncate=False)

## 3. Segment Assignment

Map `rfm_total` (range 3–12) to three campaign tiers based on thresholds configured in `config/framework_config.yaml`:

| Tier | rfm_total | Campaign Action |
|---|---|---|
| **High Value** | ≥ 9 | Priority outreach + exclusive early access |
| **Medium Value** | 6–8 | Standard campaign email + discount code |
| **Low Value** | < 6 | Reactivation message |

In [ ]:
# ── Segment Labels ────────────────────────────────────────────────────────────
HIGH_THRESHOLD   = 9
MEDIUM_THRESHOLD = 6

df_segmented = df_rfm_scored.withColumn(
    "customer_segment",
    F.when(F.col("rfm_total") >= HIGH_THRESHOLD, "High Value")
     .when(F.col("rfm_total") >= MEDIUM_THRESHOLD, "Medium Value")
     .otherwise("Low Value")
)

print("Segment distribution:")
df_segmented.groupBy("customer_segment").count().show()

print("\nCustomers with segments:")
df_segmented.select(
    "customer_id", "shipping_region", "rfm_total", "customer_segment"
).orderBy(F.col("rfm_total").desc()).show(truncate=False)

## 4. Geo Aggregation

Aggregate RFM and segment counts by `shipping_region` to inform regional targeting budgets.

In [ ]:
# ── Regional Summary ──────────────────────────────────────────────────────────
df_geo = (
    df_segmented
    .groupBy("shipping_region", "customer_segment")
    .agg(
        F.count("customer_id").alias("customer_count"),
        F.round(F.avg("total_spend"), 2).alias("avg_spend"),
        F.round(F.sum("total_spend"), 2).alias("total_regional_spend"),
    )
    .orderBy("shipping_region", F.col("customer_count").desc())
)

print("Regional segment breakdown:")
df_geo.show(truncate=False)

## 5. Audit Columns & Final Schema

Add standard audit columns (`tbl_` prefix, `ingestion_ts`, `pipeline_run_id`) as required by `framework_config.yaml`.

In [ ]:
import uuid

RUN_ID = str(uuid.uuid4())

df_output = (
    df_segmented
    .withColumnRenamed("customer_id",      "tbl_customer_id")
    .withColumnRenamed("shipping_region",   "tbl_shipping_region")
    .withColumn("tbl_rfm_total",            F.col("rfm_total"))
    .withColumn("tbl_customer_segment",     F.col("customer_segment"))
    .withColumn("tbl_ingestion_ts",         F.current_timestamp())
    .withColumn("tbl_pipeline_run_id",      F.lit(RUN_ID))
    .select(
        "tbl_customer_id", "tbl_shipping_region",
        "days_since_last_order", "order_count", "total_spend",
        "recency_score", "frequency_score", "monetary_score",
        "tbl_rfm_total", "tbl_customer_segment",
        "tbl_ingestion_ts", "tbl_pipeline_run_id",
    )
)

df_output.printSchema()
df_output.show(truncate=False)

## 6. Write to Delta Lake (GCS)

In production the output path is `gs://YOUR_PROJECT-processed/rfm_segments/`.  
The pipeline uses Delta Lake for ACID upserts with `customer_id` as the merge key.

In [ ]:
import os

OUTPUT_PATH = os.environ.get(
    "OUTPUT_PATH",
    "/tmp/amazon_campaign_rfm_output"  # local fallback
)

# ── Write (overwrite for demo, use Delta MERGE in production) ─────────────────
(
    df_output
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OUTPUT_PATH)
)

print(f"Written {df_output.count()} rows to {OUTPUT_PATH}")

# Verify read-back
spark.read.format("delta").load(OUTPUT_PATH).select(
    "tbl_customer_id", "tbl_customer_segment", "tbl_rfm_total"
).show()

## 7. Production Notes

When the Autonomous ETL Agent processes **STORY-003**, it generates a script equivalent to the cells above as `pipeline_story_003.py` and stores it in GCS.  
The generated script accepts CLI arguments:

```bash
python pipeline_story_003.py \
  --input-path  gs://YOUR_PROJECT-raw/transactions/ \
  --output-path gs://YOUR_PROJECT-processed/rfm_segments/ \
  --run-id      $RUN_ID
```

The Airflow DAG for STORY-003 (`orchestration/story_003_dag.py`) runs this job daily at 02:00 UTC via `DataprocSubmitJobOperator`.

### Segment → Campaign Mapping

| Segment | Campaign | Channel |
|---|---|---|
| High Value | `IPHONE17_PRIORITY` | Email + Push + Display |
| Medium Value | `IPHONE17_STANDARD` | Email + Display |
| Low Value | `IPHONE17_REACTIVATE` | Email only |

In [ ]:
spark.stop()
print("SparkSession stopped.")